# Show the Subglacial Plumes at 79NG

Notebook by Markus Reinert (IOW, 2024–2026, https://orcid.org/0000-0002-3761-8029).

In [ ]:
import gsw
import numpy as np
import xarray as xr
import cmocean.cm as cmo
import matplotlib.pyplot as plt

from tools.visualization import cm

## Load the model result

In [ ]:
ds = xr.open_dataset("output/out_mean_compressed.nc").set_coords("zc")

# Convert single time value from coordinate to attribute
assert ds.time.size == 1
time_value = ds.time.data[0]
ds = ds.squeeze(dim="time", drop=True)
ds.attrs["time"] = time_value

# Add (missing) geometry variables to the dataset
with xr.open_dataset("output/out_geometry_2d.nc") as geo:
    for var in geo.variables:
        if var in ds:
            print(var, "exists in dataset and geometry.")
            # Check that the data matches between dataset and geometry
            xr.testing.assert_equal(ds[var], geo[var])
        else:
            ds[var] = geo[var]

# Compute the masks
ds["mask"] = ds.bathymetry.notnull()
ds.mask.attrs = {"long_name": "ocean mask"}
ds["ice_mask"] = ds.glIceD > 0
ds.ice_mask.attrs = {"long_name": "ice mask"}

# Compute the flow speed
assert ds.velx3d.units == ds.vely3d.units, "units of u3d and v3d differ"
ds["vel3d"] = np.sqrt(ds.velx3d**2 + ds.vely3d**2)
ds.vel3d.attrs = {"long_name": "absolute value of the global velocity (3D)", "units": ds.velx3d.units}

# Check grid spacing
dlat = 1 / 240
dlon = 3 / 120
assert np.allclose(np.diff(ds.latc), dlat)
assert np.allclose(np.diff(ds.lonc), dlon)

# Use this aspect ratio in lat–lon plots for squared grid cells; since dx and dy
# are approximately equal, this gives an approximate equal-area map.
grille_carree = dlon / dlat

# Update the units for better labels with xarray
ds.latc.attrs.update({"units": "°N"})
ds.lonc.attrs.update({"units": "°E"})
ds.temp.attrs.update({"units": "°C"})
ds.salt.attrs.update({"units": "g kg$^{-1}$"})
ds.velx3d.attrs.update({"units": "m s$^{-1}$"})
ds.vely3d.attrs.update({"units": "m s$^{-1}$"})

ds

## Select the cavity

In [ ]:
cavity = ds.where(ds.ice_mask, drop=True)
cavity

## Compute averages over given distances from the ice

In [ ]:
under_ice = cavity[["mask", "glIceD", "eta", "bathymetry", "D"]]
under_ice.attrs = {"title": "Current velocities and layer heights averaged over different ranges under the ice"}
for dist_min, dist_max in [(0, 5), (0, 10), (10, 20), (20, 30), (30, 40), (40, 50)]:
    # Select the model layers that are between dist_min and dist_max meter from the ice edge
    is_dist_under_ice = (
        (cavity.zc >= cavity.eta - dist_max) &
        (cavity.zc <= cavity.eta - dist_min)
    )
    h_under_ice = cavity.h.where(is_dist_under_ice)
    # Compute the average layer thickness in this range
    under_ice[f"h_{dist_min}_{dist_max}"] = h_under_ice.mean("level")
    under_ice[f"h_{dist_min}_{dist_max}"].attrs = {
        "long_name": f"{h_under_ice.long_name} averaged between {dist_min} and {dist_max} m below the ice",
        "units": h_under_ice.units,
    }
    # Weight the layers by their thickness, assigning a weight of NaN for layers outside the selected range
    # It might be more accurate to assign partial weights for layers that are not fully in the selected range.
    weights = h_under_ice / h_under_ice.mean("level")
    # Compute vertically averaged velocities in the selected range
    # It might be better to use transports instead of velocity times layer thickness,
    # but they are not included in the dataset.
    for var, name in [("velx3d", "u"), ("vely3d", "v"), ("vel3d", "vel")]:
        under_ice[f"{name}_{dist_min}_{dist_max}"] = (cavity[var] * weights).mean("level")
        under_ice[f"{name}_{dist_min}_{dist_max}"].attrs = {
            "long_name": f"{cavity[var].long_name[:-5]} averaged between {dist_min} and {dist_max} m below the ice",
            "units": cavity[var].units,
        }
# Save the dataset for future use (optional)
# filename = "output/under_ice_averages.nc"
# under_ice.to_netcdf(filename); print("Saved as", filename)
under_ice

## Show layer thickness just below the ice edge

In [ ]:
fig, axs = plt.subplots(ncols=2, sharex=True, sharey=True, constrained_layout=True, figsize=(10, 4))

ax = axs[0]
cavity.h.isel(level=-1).plot(ax=ax, cmap=cmo.amp, vmin=0)
ax.set_title("Thickness of the uppermost model layer")
ax.set_aspect(grille_carree)

ax = axs[1]
under_ice.h_0_10.plot(ax=ax, cmap=cmo.amp, vmin=0)
ax.set_title("Average layer thickness in the uppermost 10 m")
ax.set_ylabel("")
ax.set_aspect(grille_carree)

## Create the figure

In [ ]:
figsize = (18*cm, 20*cm)
fig = plt.figure(figsize=figsize, dpi=150)
fig.suptitle("Subglacial plumes below the 79° North Glacier tongue", weight="bold", y=0.995)

left = 0.09
right = 0.99
wspace_top = 0.14
wspace_bot = 0.01
top = 0.935
bot = 0.04
hspace = 0.07
width_top = (right - left - wspace_top) / 2
height_top = width_top * under_ice.latc.size / under_ice.lonc.size * figsize[0] / figsize[1]
bot_ab = top - height_top
width_bot = (right - left - 2 * wspace_bot) / 3
height_bot = (bot_ab - 2 * hspace - bot) / 2
bot_cde = bot_ab - hspace - height_bot
axs = {
    "a": fig.add_axes((left, bot_ab, width_top, height_top)),
    "b": fig.add_axes((left + width_top + wspace_top, bot_ab, width_top, height_top)),
    "c": fig.add_axes((left, bot_cde, width_bot, height_bot)),
    "d": fig.add_axes((left + width_bot + wspace_bot, bot_cde, width_bot, height_bot)),
    "e": fig.add_axes((left + 2 * (width_bot + wspace_bot), bot_cde, width_bot, height_bot)),
    "f": fig.add_axes((left, bot, width_bot, height_bot)),
    "g": fig.add_axes((left + width_bot + wspace_bot, bot, width_bot, height_bot)),
    "h": fig.add_axes((left + 2 * (width_bot + wspace_bot), bot, width_bot, height_bot)),
}

# Set the style of the plume labels
bbox = {"boxstyle": "square", "facecolor": "0.95", "alpha": 2/3, "edgecolor": "none"}
label_plume = lambda ax, i, coords: ax.text(*coords, f"p{i}", size="small", bbox=bbox)

transects = [-22.1, -20.4]


# First row
step = 2  # draw an arrow at one of every `step` grid points
sel = dict(lonc=slice(0, None, step), latc=slice(0, None, step))
for l, depth in {"a": "0_10", "b": "30_40"}.items():
    print(f"max speed at {depth:>5} m depth: {under_ice[f'vel_{depth}'].max():.2f}")
    ax = axs[l]
    # Show land in gray and the model domain in white where there are no data
    ax.set_facecolor("0.8")
    ds.mask.where(ds.mask).plot(ax=ax, vmax=2, cmap="Greys", add_colorbar=False)
    # Show the flow speed
    im = under_ice[f"vel_{depth}"].plot(ax=ax, vmin=0, vmax=0.3, cmap=cmo.amp, add_colorbar=False)
    # Show the flow direction with arrows that become gradually more transparent for speeds below 0.15 m/s
    color = np.zeros((under_ice[f"vel_{depth}"].isel(**sel).size, 4))
    color[:, -1] = np.minimum(under_ice[f"vel_{depth}"].isel(**sel).data.flatten() / 0.15, 1)
    quiver_args = (cavity.lonc[::step], cavity.latc[::step], under_ice[f"u_{depth}"].isel(**sel), under_ice[f"v_{depth}"].isel(**sel))
    ax.quiver(*quiver_args, color=color)
    ax.set(aspect=grille_carree, title=f"Flow {depth.replace('_', ' to ')} m below the ice", xlabel="", ylabel="")
    ax.xaxis.set_major_formatter(lambda x, pos: f"{-x:.0f}°W")
    ax.yaxis.set_major_formatter(lambda x, pos: f"{round(x, 10)}°N")
    # Mark the transects
    for lonc in transects:
        ax.axvline(lonc, ls=":", lw=1, color="0.5")
    if l == "a":
        # Add a colorbar
        cax = ax.inset_axes((-22.45, 79.77, 1.6, 0.02), transform=ax.transData)
        fig.colorbar(im, cax=cax, orientation="horizontal", label="flow speed in m s$^{-1}$", extend="max", ticks=np.linspace(0, 0.3, 4))

        # Show a zoom of the middle plume on the ice topography
        # Define the zoomed area
        x0, x1 = (-22.06, -21.43)
        y0, y1 = (79.40, 79.49)
        print(f"    Longitudinal extent of the zoomed area: {gsw.distance([x0, x1], [y0, y0])[0] / 1e3:.2f} km (south)")
        print(f"    Longitudinal extent of the zoomed area: {gsw.distance([x0, x1], [y1, y1])[0] / 1e3:.2f} km (north)")
        print(f"    Meridional   extent of the zoomed area: {gsw.distance([x0, x0], [y0, y1])[0] / 1e3:.2f} km")
        # Define the position of the zoomed axes
        x_zoom = -21.1
        y_zoom = 79.275
        h_zoom = 0.15
        w_zoom = h_zoom  / (y1 - y0) * (x1 - x0)
        # Mark the zoomed area in the main plot
        lw = plt.rcParams["axes.linewidth"]
        ax.plot([x0, x1, x1, x0, x0], [y0, y0, y1, y1, y0], "k", lw=lw)
        # Connect the zoomed area with the zoomed axes
        for xpair, ypair in [[(x0, x_zoom), (y0, y_zoom)], [(x1, x_zoom + w_zoom), (y1, y_zoom + h_zoom)]]:
            ax.plot(xpair, ypair, "k", lw=lw, ls="--")
        # Create the zoomed axes
        axin = ax.inset_axes((x_zoom, y_zoom, w_zoom, h_zoom), transform=ax.transData)
        im = (-ds.eta).plot.contourf(ax=axin, cmap=cmo.ice_r, levels=np.linspace(200, 330, 14), add_colorbar=False)
        # Add a colorbar
        with plt.rc_context({"ytick.labelsize": "small", "axes.labelsize": "small", "ytick.major.size": 2, "ytick.major.pad": 0}):
            cax = ax.inset_axes((x_zoom + w_zoom + 0.04, y_zoom, 0.04, h_zoom), transform=ax.transData)
            cbar = fig.colorbar(im, cax, ticks=[200, 250, 300], format="{x:.0f} m")
            cbar.set_label("ice draft", labelpad=1, loc="bottom")
            cax.set_yticks([], minor=True)
        axin.quiver(*quiver_args, color="#fd3c06", scale=1.5, width=0.01)  # xkcd color red orange
        axin.set(aspect=grille_carree, xlim=(x0, x1), ylim=(y0, y1), title="", xlabel="", ylabel="", xticks=[], yticks=[])

        label_plume(ax, 1, (-22.55, 79.285))
        label_plume(ax, 2, (-22.50, 79.360))
        label_plume(ax, 3, (-22.60, 79.470))
    else:
        label_plume(ax, 1, (-20.82, 79.440))
        # plume 2 is absent in the deep layer
        label_plume(ax, 3, (-21.60, 79.625))

        # Label the transects
        for i, lonc in enumerate(transects):
            ax.text(lonc + 0.02, ax.get_ylim()[0] + 0.01, f"transect T{i + 1}", va="bottom")


# Second and third row
for row in range(2):
    ds_slice = ds.sel(lonc=transects[row], method="nearest").sel(latc=slice(79.77)).dropna(dim="latc", subset=["bathymetry"])
    for ax, cmap, levels, var, name, label in [
        [axs["cf"[row]], cmo.balance, np.linspace(-0.3, 0.3, 21), "velx3d", "Velocity", "zonal velocity"],
        [axs["dg"[row]], cmo.thermal, np.linspace(-1.0, 1.5, 21), "temp", "Temperature", "temperature"],
        [axs["eh"[row]], cmo.haline, np.linspace(33.0, 34.4, 21), "salt", "Salinity", "salinity"],
    ]:
        data = ds_slice[var]
        print(f"{var} range: {data.min():.3f} to {data.max():.3f} {data.units}")
        # Even though there are no velocities below -0.3 m/, we extend the colorbar in both directions,
        # because otherwise the velocity colormap would not be symmetric about zero.
        im = data.plot.contourf(ax=ax, y="zc", levels=levels, cmap=cmap, extend="both", add_colorbar=False)
        if var == "velx3d":
            im_vel = im
            if row == 0:
                # plume 1 is labeled in the inset
                label_plume(ax, 2, (79.405, -280))
                label_plume(ax, 3, (79.510, -320))
            else:
                label_plume(ax, 1, (79.46, -230))
                # plume 2 is absent in transect 2
                label_plume(ax, 3, (79.71, -150))

                # Mark the 30-40 m range
                (ds_slice.eta - 30).plot(ax=ax, ls=":", color="k", lw=0.5)
                (ds_slice.eta - 40).plot(ax=ax, ls=":", color="k", lw=0.5)
                # Annotate the detached plume
                ax.annotate(
                    "max. vel. detached from the ice",
                    (79.47, -120),
                    (79.46, -45),
                    size="small", va="top",
                    arrowprops={"arrowstyle": "->", "shrinkA": 0, "shrinkB": 0, "relpos": (0.1, 0), "linewidth": 0.7},
                )
            ax.set_ylabel("depth in m")
            ax.yaxis.set_major_formatter(lambda x, pos: f"{-x:.0f}")
        else:
            if row == 1:
                # Show selected velocity contours with the same colors as in the velocity plot
                vel_levels = [-0.1, 0.1]
                colors = list(im_vel.cmap(im_vel.norm(vel_levels)))
                ds_slice.velx3d.plot.contour(ax=ax, y="zc", levels=vel_levels, colors=colors, linewidths=0.5, linestyles="--")
            ax.set_ylabel("")
            ax.yaxis.set_major_formatter(lambda x, pos: "")
        ax.set(title=f"{name} at T{row + 1}", xlabel="", ylim=(-500, -40))
        ax.xaxis.set_major_formatter(lambda x, pos: f"{round(x, 10)}°N")        
        if row == 0:
            # Add a colorbar
            cax = ax.inset_axes((79.41, -100, 0.11, 20), transform=ax.transData)
            cbar = fig.colorbar(im, cax=cax, orientation="horizontal", ticks=levels[::10], label=f"{label}\nin {data.units}")
            cbar.set_ticks([], minor=True)

            # Zoom to a subglacial channel
            # Define the zoomed area
            x0, x1 = (79.35 + dlat/2 - 6 * dlat, 79.35 + dlat/2)
            y0, y1 = (-420, -315)
            # Define the position of the zoomed axes
            x_zoom = 79.31
            y_zoom = -290
            w_zoom = 0.08
            h_zoom = 240
            # Mark the zoomed area in the main plot
            lw = plt.rcParams["axes.linewidth"]
            ax.plot([x0, x1, x1, x0, x0], [y0, y0, y1, y1, y0], "k", lw=lw)
            # Connect the zoomed area with the zoomed axes
            for xpair, ypair in [
                [(x0, x_zoom), (y0, y_zoom)],
                [(x1, x_zoom + w_zoom), (y0, y_zoom)],
                [(x0, x_zoom), (y1, y_zoom + h_zoom)],
                [(x1, x_zoom + w_zoom), (y1, y_zoom + h_zoom)],
            ]:
                ax.plot(xpair, ypair, "k", lw=lw, ls="--")
            # Create the zoomed axes
            axin = ax.inset_axes((x_zoom, y_zoom, w_zoom, h_zoom), transform=ax.transData)
            data.plot(ax=axin, y="zc", vmin=levels[0], vmax=levels[-1], cmap=cmap, add_colorbar=False)
            axin.set(title="", xlabel="", ylabel="", xlim=(x0, x1), ylim=(y0, y1), xticks=[], yticks=[])
            if var == "velx3d":
                label_plume(axin, 1, (x0 + dlat, -335))


for l, ax in axs.items():
    ax.set_title(f"({l}) {ax.get_title()}")

fig.savefig("figures/fig_4_subglacial_plumes.png", dpi=600)